In [20]:
from google.colab import files
uploaded = files.upload()



Saving ratings.xlsx to ratings.xlsx


In [21]:
import pandas as pd
import numpy as np

df = pd.read_excel('ratings.xlsx')
print(df)

  Unnamed: 0  Forrest Gump  Ray  Mrs. Doubtfire  The Incredibles  \
0     Esther           5.0  4.0             4.5              4.5   
1     Alicia           3.0  5.0             4.0              4.0   
2    Jessica           3.5  3.5             NaN              NaN   
3     Amanda           NaN  NaN             NaN              4.0   
4      Chris           4.5  4.5             3.0              5.0   

   Fifty Shades of Grey  Wolf on Wall Street  
0                   3.0                  4.0  
1                   3.0                  3.5  
2                   NaN                  3.5  
3                   4.0                  5.0  
4                   NaN                  5.0  


In [22]:
df.to_csv('ratings.csv')
print(df.to_string())

  Unnamed: 0  Forrest Gump  Ray  Mrs. Doubtfire  The Incredibles  Fifty Shades of Grey  Wolf on Wall Street
0     Esther           5.0  4.0             4.5              4.5                   3.0                  4.0
1     Alicia           3.0  5.0             4.0              4.0                   3.0                  3.5
2    Jessica           3.5  3.5             NaN              NaN                   NaN                  3.5
3     Amanda           NaN  NaN             NaN              4.0                   4.0                  5.0
4      Chris           4.5  4.5             3.0              5.0                   NaN                  5.0


upload and read file. convert xlsx to csv.

In [24]:
df = df.set_index('Unnamed: 0')
avg_per_user = df.mean(axis=1)
avg_per_movie = df.mean(axis=0)
print("Average rating given by each user:")
for user, val in avg_per_user.items():
    print(f"   {user:<10} {val:.2f}")

print("Average rating received by each movie:")
for movie, val in avg_per_movie.items():
    print(f"   {movie:<20} {val:.2f}")

Average rating GIVEN by each user:
   Esther     4.17
   Alicia     3.75
   Jessica    3.50
   Amanda     4.33
   Chris      4.40
Average rating RECEIVED by each movie:
   Forrest Gump         4.00
   Ray                  4.25
   Mrs. Doubtfire       3.83
   The Incredibles      4.38
   Fifty Shades of Grey 3.33
   Wolf on Wall Street  4.20


use the 'Unnamed: 0' column as the row labels. calculate the average across rows and columns. use for loop and .items() to go through each user and their average value one by one. Format data with 2 decimal places.

In [25]:
def normalize_row(row):
    row_min = row.min()
    row_max = row.max()
    if row_max == row_min:
        return row.apply(lambda x: 0.5 if pd.notna(x) else np.nan)
    return (row - row_min) / (row_max - row_min)

df_norm = df.apply(normalize_row, axis=1)

find lowest and highest rating. if they are equal, return 0.5 for each value, otherwise it normalizes the numbers between 0 and 1

In [32]:
print("Normalized Ratings DataFrame (0–1 scale per user) ──")
print(df_norm.round(3).to_string())

norm_avg_per_user  = df_norm.mean(axis=1)
norm_avg_per_movie = df_norm.mean(axis=0)

print("Average normalized rating given by each user:")
for user, val in norm_avg_per_user.items():
    print(f"   {user:<10} {val:.3f}")

print("Average normalized rating received by each movie:")
for movie, val in norm_avg_per_movie.items():
    print(f"   {movie:<20} {val:.3f}")


Normalized Ratings DataFrame (0–1 scale per user) ──
            Forrest Gump   Ray  Mrs. Doubtfire  The Incredibles  Fifty Shades of Grey  Wolf on Wall Street
Unnamed: 0                                                                                                
Esther              1.00  0.50            0.75             0.75                   0.0                 0.50
Alicia              0.00  1.00            0.50             0.50                   0.0                 0.25
Jessica             0.50  0.50             NaN              NaN                   NaN                 0.50
Amanda               NaN   NaN             NaN             0.00                   0.0                 1.00
Chris               0.75  0.75            0.00             1.00                   NaN                 1.00
Average normalized rating given by each user:
   Esther     0.583
   Alicia     0.375
   Jessica    0.500
   Amanda     0.333
   Chris      0.700
Average normalized rating received by each movie:
  

print from normalized ratings table, round to 3 decimal places.

In [33]:
comparison = pd.DataFrame({
    'Actual Avg':     avg_per_movie.round(2),
    'Normalized Avg': norm_avg_per_movie.round(3),
})
print(comparison.to_string())

                      Actual Avg  Normalized Avg
Forrest Gump                4.00           0.562
Ray                         4.25           0.688
Mrs. Doubtfire              3.83           0.417
The Incredibles             4.38           0.562
Fifty Shades of Grey        3.33           0.000
Wolf on Wall Street         4.20           0.650


create new dataframe that shows actual and normalized avg.

In [38]:
conclusion = """

Normalized ratings turn everyone’s rating scale into a standard 0 to 1 scale.

ADVANTAGES of normalized ratings
─────────────────────────────────
1. Corrects for rating-scale bias.
   Some users are "generous" raters (like Esther, Chris, who rarely go
   below 3.5) while others are "harsh" (like Alicia, who uses the full
   1–5 range). Normalization puts everyone on the same page,
   so a movie ranked highly by a stingy rater gets proper credit.

2. Highlights preferences.
   Instead of looking at the raw number, it identifies which films are a user's
   relative favorites.

3. More fair comparisons across users with missing data.
   Raw averages can be misleading; normalization makes user
   rankings comparable.

DISADVANTAGES of normalized ratings
──────────────────────────────────────
1. Loses quality information.
   A film that everyone rated 2/5 (genuinely bad) can end up with
   a high normalized score if it happened to be the best film a
   particular user saw. Normalization hides unanimous low scores.

2. Sensitive to the range of ratings.
   If a user rated only two films — one a 1 and one a 5 — those
   become 0.0 and 1.0. A user who rated six films more carefully
   produces a richer, more reliable normalized profile.

3. Harder to interpret.
   A normalized score of 0.67 is not intuitively meaningful to
   most people, whereas "4 out of 5" is.

4. Small sample distortion.
   With only 5 users and 6 movies, normalization can exaggerate
   small differences.

Normalized ratings are best for personalized recommendations because they
focus on what a user likes compared to their own history.
Actual ratings are best for public review sites because they show a movie's
overall quality on a simple, familiar scale.
══════════════════════════════════════════════════════════════════
"""
print(conclusion)




Normalized ratings turn everyone’s rating scale into a standard 0 to 1 scale.

ADVANTAGES of normalized ratings
─────────────────────────────────
1. Corrects for rating-scale bias.
   Some users are "generous" raters (like Esther, Chris, who rarely go
   below 3.5) while others are "harsh" (like Alicia, who uses the full
   1–5 range). Normalization puts everyone on the same page,
   so a movie ranked highly by a stingy rater gets proper credit.

2. Highlights preferences.
   Instead of looking at the raw number, it identifies which films are a user's
   relative favorites.

3. More fair comparisons across users with missing data.
   Raw averages can be misleading; normalization makes user
   rankings comparable.

DISADVANTAGES of normalized ratings
──────────────────────────────────────
1. Loses quality information.
   A film that everyone rated 2/5 (genuinely bad) can end up with
   a high normalized score if it happened to be the best film a
   particular user saw. Normalization h